In [ ]:
import os
import sys

# Dynamically find the project root (assumes notebook is in docs/notebooks/)
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, "..", ".."))

# Define standard data paths
RAW_AUDIO_DIR = os.path.join(PROJECT_ROOT, "docs", "data", "raw", "family_voice")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "docs", "data", "processed", "family_voice")
WAV_STORAGE_DIR = os.path.join(PROJECT_ROOT, "docs", "data", "wav")
VIDEO_INPUT_DIR = os.path.join(PROJECT_ROOT, "videos")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "docs", "results", "family_voice_vit")

# Ensure directories exist
os.makedirs(WAV_STORAGE_DIR, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"WAV Storage: {WAV_STORAGE_DIR}")

In [ ]:
from moviepy import VideoFileClip

# Replace with your actual filename
video = VideoFileClip(f"{PROJECT_ROOT}/videos/raw_video_mine/PXL_20241017_025315117.mp4")
video.audio.write_audiofile(f"{WAV_STORAGE_DIR}/full_speech.wav")

In [ ]:
import librosa
import soundfile as sf
import os

audio_path = f"{WAV_STORAGE_DIR}/full_speech.wav"
output_base = f"{PROJECT_ROOT}/docs/data/raw/family_voice/mine"
os.makedirs(output_base, exist_ok=True)



y, sr = librosa.load(audio_path)
segment_len = 6 * sr # 3 seconds

for i, start in enumerate(range(0, len(y), segment_len)):
    chunk = y[start : start + segment_len]
    if len(chunk) == segment_len:
        sf.write(f"{output_base}/segment_{i:03d}.wav", chunk, sr)

In [ ]:
import librosa
import soundfile as sf
import os
from moviepy import VideoFileClip
import glob

# --- CONFIG ---
# Folder where your raw videos/audio of your son are kept
INPUT_FOLDER = os.path.join(VIDEO_INPUT_DIR, "raw_videos_son")
# Folder where the final 6s clips should go
OUTPUT_BASE = os.path.join(RAW_AUDIO_DIR, "son")
SECONDS = 6  # Matches your training duration

os.makedirs(OUTPUT_BASE, exist_ok=True)

# Find all mp4 and wav files
files_to_process = glob.glob(os.path.join(INPUT_FOLDER, "*.mp4")) + glob.glob(os.path.join(INPUT_FOLDER, "*.wav"))

print(f"Found {len(files_to_process)} files to process.")

for file_path in files_to_process:
    file_name = os.path.basename(file_path)
    file_base = os.path.splitext(file_name)[0]
    
    print(f"--- Processing: {file_name} ---")
    
    # 1. Handle MP4 extraction if necessary
    if file_path.endswith(".mp4"):
        temp_wav = "temp_batch.wav"
        video = VideoFileClip(file_path)
        video.audio.write_audiofile(temp_wav, logger=None)
        current_audio = temp_wav
    else:
        current_audio = file_path

    # 2. Load and Slice
    try:
        y, sr = librosa.load(current_audio)
        segment_samples = SECONDS * sr
        
        for i, start in enumerate(range(0, len(y), segment_samples)):
            chunk = y[start : start + segment_samples]
            
            # Ensure we only save full 6-second segments
            if len(chunk) == segment_samples:
                # Unique name: OriginalFilename_seg_001.wav
                out_name = f"{file_base}_seg_{i:03d}.wav"
                save_path = os.path.join(OUTPUT_BASE, out_name)
                
                sf.write(save_path, chunk, sr)
        
        print(f"Successfully sliced {file_name}")
        
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

# Cleanup temp file
if os.path.exists("temp_batch.wav"):
    os.remove("temp_batch.wav")

print(f"\n✅ Done! All clips are in: {OUTPUT_BASE}")

In [ ]:
from moviepy import VideoFileClip

# Replace with your actual filename
video = VideoFileClip(f"{PROJECT_ROOT}/videos/raw_vide_daughter/niralya_biggerisbetter.mp4")
video.audio.write_audiofile(f"{WAV_STORAGE_DIR}/niralya_biggerisbetter.wav")

In [ ]:
import librosa
import soundfile as sf
import os

audio_path = f"{WAV_STORAGE_DIR}/niralya_biggerisbetter.wav"
output_base = f"{PROJECT_ROOT}/docs/data/raw/family_voice/daughter"

os.makedirs(output_base, exist_ok=True)



y, sr = librosa.load(audio_path)
segment_len = 6 * sr # 3 seconds

for i, start in enumerate(range(0, len(y), segment_len)):
    chunk = y[start : start + segment_len]
    if len(chunk) == segment_len:
        sf.write(f"{output_base}/segment_{i:03d}.wav", chunk, sr)

In [ ]:
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- SETTINGS ---
RAW_DIR = RAW_AUDIO_DIR
PROCESSED_DIR = PROCESSED_DATA_DIR
def create_spectrograms():
    # Loop through each family member's folder (Me, Daughter, Son)
    for person_folder in os.listdir(RAW_DIR):
        person_path = os.path.join(RAW_DIR, person_folder)
        if not os.path.isdir(person_path): continue
        
        # Create corresponding folder in 'processed'
        output_folder = os.path.join(PROCESSED_DIR, person_folder)
        os.makedirs(output_folder, exist_ok=True)
        
        print(f"Processing voice clips for: {person_folder}...")
        
        for audio_file in os.listdir(person_path):
            if audio_file.endswith(".wav"):
                try:
                    # 1. Load audio
                    y, sr = librosa.load(os.path.join(person_path, audio_file), duration=6.0)
                    
                    # 2. Generate Mel-spectrogram
                    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
                    S_db = librosa.power_to_db(S, ref=np.max)
                    
                    # 3. Save as Image (224x224 for ViT)
                    fig = plt.figure(figsize=(2.24, 2.24), dpi=100)
                    ax = fig.add_axes([0, 0, 1, 1])
                    ax.axis('off')
                    librosa.display.specshow(S_db, sr=sr)
                    
                    output_name = audio_file.replace(".wav", ".png")
                    plt.savefig(os.path.join(output_folder, output_name), bbox_inches='tight', pad_inches=0)
                    plt.close(fig)
                except Exception as e:
                    print(f"Error processing {audio_file}: {e}")

if __name__ == "__main__":
    create_spectrograms()
    print("\n✅ All done! Your audio is now converted to images.")

In [ ]:
from moviepy import VideoFileClip

# Replace with your actual filename test audio file
video = VideoFileClip(f"{PROJECT_ROOT}/videos/test/feelings.mp4")
video.audio.write_audiofile(f"{WAV_STORAGE_DIR}/son_mine_feeling.wav")

In [ ]:
from moviepy import VideoFileClip

# Replace with your actual filename test audio file
video = VideoFileClip(f"{PROJECT_ROOT}/videos/test/happybirthday.mp4")
video.audio.write_audiofile(f"{WAV_STORAGE_DIR}/happy_birthday.wav")

In [ ]:
from moviepy import VideoFileClip

# Replace with your actual filename test audio file
video = VideoFileClip(f"{PROJECT_ROOT}/videos/test/happybirthday.mp4")
video.audio.write_audiofile(f"{WAV_STORAGE_DIR}/happy_birthday.wav")

In [64]:
from moviepy import VideoFileClip

# Replace with your actual filename test audio file
video = VideoFileClip(f"{PROJECT_ROOT}/videos/test/tamil_talk.mp4")
video.audio.write_audiofile(f"{WAV_STORAGE_DIR}/tamil_talk.wav")

MoviePy - Writing audio in /Users/sangeetha/Supportvector2026/llm/projects/labs/vision_transformers_vs_cnn/docs/data/wav/tamil_talk.wav


MoviePy - Done.


In [63]:
import torch
import librosa
import numpy as np
import joblib
import matplotlib.pyplot as plt
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification

# --- SETTINGS ---
MODEL_PATH = os.path.join(RESULTS_DIR, "final_model")
LABEL_ENCODER_PATH = os.path.join(RESULTS_DIR, "label_encoder.joblib")
# Update this to a NEW voice clip you want to test!
#TEST_AUDIO_PATH = f"{WAV_STORAGE_DIR}/eathquake.wav" 
#TEST_AUDIO_PATH = f"{WAV_STORAGE_DIR}/son_mine_feeling.wav" 
TEST_AUDIO_PATH = f"{WAV_STORAGE_DIR}/tamil_talk.wav" 
#{PROJECT_ROOT}/videos/test/feelings.mp4
def predict_voice(audio_path):
    # 1. Load Label Encoder and Processor
    label_encoder = joblib.load(LABEL_ENCODER_PATH)
    processor = ViTImageProcessor.from_pretrained(MODEL_PATH)
    
    # 2. Load the Trained Model
    model = ViTForImageClassification.from_pretrained(MODEL_PATH)
    model.eval() # Set to evaluation mode

    # 3. Process Audio to Spectrogram Image (Internal Conversion)
    y, sr = librosa.load(audio_path, duration=6.0)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    
    # Save temp image to process (matches training pipeline)
    temp_img_path = "temp_inference.png"
    plt.figure(figsize=(2.24, 2.24), dpi=100)
    plt.axis('off')
    librosa.display.specshow(S_db, sr=sr)
    plt.savefig(temp_img_path, bbox_inches='tight', pad_inches=0)
    plt.close()

    # 4. Prepare Image for ViT
    image = Image.open(temp_img_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")

    # 5. Inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Convert raw scores to percentages (probabilities)
        probs = torch.nn.functional.softmax(logits, dim=-1)[0]

    # 6. Map back to ALL Names
    # label_encoder.classes_ gives us ['Daughter', 'Me', 'Son'] in order
    all_labels = label_encoder.classes_
    
    print(f"\n--- VOICE ANALYSIS BREAKDOWN ---")
    
    # Zip names and scores together and sort by highest percentage
    results = sorted(
        zip(all_labels, probs.tolist()), 
        key=lambda x: x[1], 
        reverse=True
    )

    for name, score in results:
        # Create a small visual bar to see the dominance
        bar = "█" * int(score * 20)
        print(f"{name:10}: {score:6.2%} {bar}")
        
    print(f"--------------------------------\n")

    # THIS IS THE MISSING PART:
if __name__ == "__main__":
    # Make sure this variable points to your actual test file
    print(f"Processing: {TEST_AUDIO_PATH}...")
    predict_voice(TEST_AUDIO_PATH)

Processing: /Users/sangeetha/Supportvector2026/llm/projects/labs/vision_transformers_vs_cnn/docs/data/wav/tamil_talk.wav...

--- VOICE ANALYSIS BREAKDOWN ---
son       : 98.22% ███████████████████
mine      :  0.90% 
daughter  :  0.87% 
--------------------------------

